# Atividade 2.1 - Zeros de Funções Reais (Questão A)

**Objetivo:** Comparação de métodos numéricos (Bissecção, Posição Falsa, Ponto Fixo, Newton e Secante) para encontrar raízes de funções reais.

Reprodução das tabelas dos **Exemplos 18, 19, 20, 21 e 22** conforme solicitado, seguindo os critérios de parada $\epsilon_1, \epsilon_2$ e número máximo de iterações.

In [ ]:
import math
import time
import pandas as pd
import numpy as np

#configuração para as tabelas ficarem mais bonitas e legíveis
pd.set_option('display.float_format', '{:.8f}'.format)
pd.set_option('display.colheader_justify', 'center')

#funcoees dos Métodos Numéricos

def metodo_bisseccao(f, a, b, eps, itmax=100):
    start = time.perf_counter()
    if f(a) * f(b) > 0: return {'Status': 'Falha (Sinais iguais)'}
    
    delta_x = abs(b - a)
    k = 0
    x = a
    fx = f(x)
    
    while k < itmax:
        delta_x = delta_x / 2
        x = a + delta_x
        fx = f(x)
        
        if (delta_x <= eps) or (abs(fx) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x, 'Erro f(x)': abs(fx), 'Tempo (s)': end-start}
            
        if f(a) * fx < 0:
            b = x
        else:
            a = x
        k += 1
    
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(fx), 'Tempo (s)': end-start}

def metodo_posicao_falsa(f, a, b, eps, itmax=100):
    start = time.perf_counter()
    if f(a) * f(b) > 0: return {'Status': 'Falha (Sinais iguais)'}
    
    k = 0
    while k < itmax:
        if abs(b - a) <= eps: break
        
        fa, fb = f(a), f(b)
        if (fb - fa) == 0: return {'Status': 'Erro: Divisão por zero'}
        
        x = (a * fb - b * fa) / (fb - fa)
        fx = f(x)
        
        if abs(fx) <= eps:
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x, 'Erro f(x)': abs(fx), 'Tempo (s)': end-start}
            
        if fa * fx < 0:
            b = x
        else:
            a = x
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': end-start}

def metodo_mpf(f, phi, x0, eps, itmax=100):
    start = time.perf_counter()
    x = x0
    k = 0
    
    if abs(f(x)) <= eps:
        return {'k (Iter)': 0, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': 0}
        
    while k < itmax:
        try:
            x_new = phi(x)
        except:
             return {'Status': 'Erro de Domínio'}

        if (abs(x_new - x) <= eps) or (abs(f(x_new)) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x_new, 'Erro f(x)': abs(f(x_new)), 'Tempo (s)': end-start}
            
        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': end-start}

def metodo_newton(f, df, x0, eps, itmax=100):
    start = time.perf_counter()
    x = x0
    k = 0
    
    if abs(f(x)) <= eps:
        return {'k (Iter)': 0, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': 0}
        
    while k < itmax:
        deriv = df(x)
        if deriv == 0: return {'Status': 'Erro: Derivada zero'}
        
        x_new = x - (f(x) / deriv)
        
        if (abs(x_new - x) <= eps) or (abs(f(x_new)) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x_new, 'Erro f(x)': abs(f(x_new)), 'Tempo (s)': end-start}
            
        x = x_new
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x, 'Erro f(x)': abs(f(x)), 'Tempo (s)': end-start}

def metodo_secante(f, x0, x1, eps, itmax=100):
    start = time.perf_counter()
    fx0, fx1 = f(x0), f(x1)
    
    if abs(fx0) <= eps: return {'k (Iter)': 0, 'Raiz Aproximada': x0, 'Erro f(x)': abs(fx0), 'Tempo (s)': 0}
    if abs(fx1) <= eps: return {'k (Iter)': 0, 'Raiz Aproximada': x1, 'Erro f(x)': abs(fx1), 'Tempo (s)': 0}
    
    k = 0
    while k < itmax:
        den = fx1 - fx0
        if den == 0: return {'Status': 'Erro: Divisão por zero'}
        
        x_new = x1 - (fx1 * (x1 - x0) / den)
        fx_new = f(x_new)
        
        if (abs(x_new - x1) <= eps) or (abs(fx_new) <= eps):
            end = time.perf_counter()
            return {'k (Iter)': k+1, 'Raiz Aproximada': x_new, 'Erro f(x)': abs(fx_new), 'Tempo (s)': end-start}
            
        x0, fx0 = x1, fx1
        x1, fx1 = x_new, fx_new
        k += 1
        
    end = time.perf_counter()
    return {'k (Iter)': k, 'Raiz Aproximada': x1, 'Erro f(x)': abs(fx1), 'Tempo (s)': end-start}

def executar_bonito(titulo, f, df, phi, a, b, x0, x1_sec, eps):
    resultados = []
    
    #executa cada metodo e coleta resultados
    #bisseccão
    r = metodo_bisseccao(f, a, b, eps)
    if r: resultados.append(['Bissecção'] + list(r.values())[:4]) # Pega apenas os 4 primeiros valores
    
    #posicao falsa
    r = metodo_posicao_falsa(f, a, b, eps)
    if r: resultados.append(['Posição Falsa'] + list(r.values())[:4])
    
    #MPF
    r = metodo_mpf(f, phi, x0, eps)
    if r: resultados.append(['MPF'] + list(r.values())[:4])
    
    #Newton
    r = metodo_newton(f, df, x0, eps)
    if r: resultados.append(['Newton'] + list(r.values())[:4])
    
    #secante
    r = metodo_secante(f, x0, x1_sec, eps)
    if r: resultados.append(['Secante'] + list(r.values())[:4])
    
    #cria tabela formatada
    df_res = pd.DataFrame(resultados, columns=['Método', 'Iterações (k)', 'Raiz Encontrada', 'Erro Abs |f(x)|', 'Tempo (s)'])
    
    #exibe a tabela
    print("\n" + "="*60)
    print(f"{titulo.center(60)}")
    print(f"Intervalo: [{a}, {b}] | Precisão: {eps}")
    print("="*60)
    display(df_res)

In [ ]:
#EXEMPLO 18
#f(x) = e^(-x^2) - cos(x)
#Intervalo: [1, 2]; Eps = 10^-4
#Phi(x) = arccos(e^(-x^2))

def f18(x): 
    return math.exp(-x**2) - math.cos(x)

def df18(x): 
    #derivada: -2x*e^(-x^2) + sen(x)
    return -2*x*math.exp(-x**2) + math.sin(x)

def phi18(x): 
    #MPF: x = arccos(e^(-x^2))
    try:
        val = math.exp(-x**2)
        #protecao de domínio para arccos [-1, 1]
        if val > 1: val = 1
        if val < -1: val = -1
        return math.acos(val)
    except: return x

#parametros do exemplo
a, b = 1.0, 2.0
eps = 1e-4
x0 = 1.5      #ponto medio
x1_sec = 2.0  #extremidade para secante

executar_bonito("EXEMPLO 18", f18, df18, phi18, a, b, x0, x1_sec, eps)


                         EXEMPLO 18                         
Intervalo: [1.0, 2.0] | Precisão: 0.0001


,Método,Iterações (k),Raiz Encontrada,Erro Abs |f(x)|,Tempo (s)
0,Bissecção,9,1.44726562,0.00009455,0.00001480
1,Posição Falsa,6,1.44735707,0.00003639,0.00000810
2,MPF,6,1.44751711,0.00006542,0.00000710
3,Newton,2,1.44741635,0.00000132,0.00000500
4,Secante,3,1.44742556,0.00000718,0.00000370


In [ ]:
# EXEMPLO 19
# f(x) = x^3 - x - 1
def f19(x): return x**3 - x - 1
def df19(x): return 3*x**2 - 1
def phi19(x): return math.pow(x + 1, 1/3) # x = (x+1)^(1/3)

#parametros: intervalo [1, 2], Eps = 10^-6
#chute inicial (x0): ponto médio (1.5)
executar_bonito("EXEMPLO 19", f19, df19, phi19, 1.0, 2.0, 1.5, 2.0, 1e-6)


                         EXEMPLO 19                         
Intervalo: [1.0, 2.0] | Precisão: 1e-06


,Método,Iterações (k),Raiz Encontrada,Erro Abs |f(x)|,Tempo (s)
0,Bissecção,20,1.32471752,0.00000186,0.00001200
1,Posição Falsa,17,1.32471776,0.00000083,0.00001170
2,MPF,9,1.32471801,0.00000023,0.00000650
3,Newton,3,1.32471817,0.00000092,0.00000370
4,Secante,5,1.32471803,0.00000031,0.00000330


In [ ]:
#EXEMPLO 20
#f(x) = 4sen(x) - e^x
def f20(x): return 4 * math.sin(x) - math.exp(x)
def df20(x): return 4 * math.cos(x) - math.exp(x)
def phi20(x): 
    #isolando x: 4sen(x) = e^x => sen(x) = e^x / 4 => x = arcsen(e^x / 4)
    try:
        val = math.exp(x) / 4
        return math.asin(val)
    except: return x 

#parametros: intervalo [0, 1], eps = 10^-5
executar_bonito("EXEMPLO 20", f20, df20, phi20, 0.0, 1.0, 0.5, 1.0, 1e-5)


                         EXEMPLO 20                         
Intervalo: [0.0, 1.0] | Precisão: 1e-05


,Método,Iterações (k),Raiz Encontrada,Erro Abs |f(x)|,Tempo (s)
0,Bissecção,16,0.37055969,0.00000364,0.00001330
1,Posição Falsa,8,0.37055883,0.00000167,0.00000730
2,MPF,11,0.37056258,0.00001022,0.00000660
3,Newton,3,0.37055808,0.00000003,0.00000420
4,Secante,6,0.37055823,0.00000030,0.00000370


In [ ]:
#EXEMPLO 21
#f(x) = x * log10(x) - 1
def f21(x): return x * math.log10(x) - 1
def df21(x): return math.log10(x) + (1 / math.log(10)) # Regra do produto
def phi21(x): 
    #isolando x: log10(x) = 1/x => x = 10^(1/x)
    try: return math.pow(10, 1/x)
    except: return x

#parametros: intervalo [2, 3], Eps = 10^-7
executar_bonito("EXEMPLO 21", f21, df21, phi21, 2.0, 3.0, 2.5, 3.0, 1e-7)


                         EXEMPLO 21                         
Intervalo: [2.0, 3.0] | Precisão: 1e-07


,Método,Iterações (k),Raiz Encontrada,Erro Abs |f(x)|,Tempo (s)
0,Bissecção,Falha (Sinais iguais),NaN,NaN,NaN
1,Posição Falsa,Falha (Sinais iguais),NaN,NaN,NaN
2,MPF,29,1.76322279,0.00000006,0.00001200
3,Newton,4,1.76322283,0.00000000,0.00000380
4,Secante,5,1.76322283,0.00000000,0.00000280


In [ ]:
#EXEMPLO 22
#f(x) = (x - 1)^2 * (x - 1.5)
#isso expande para: x^3 - 3.5x^2 + 4x - 1.5
#usei a forma fatorada para que eu pudesse garantir uma certa precisão matemática (evitando o erro de digitação 3.2 vs 3.5)

def f22(x): 
    return (x - 1)**2 * (x - 1.5)

def df22(x): 
    #derivada pela regra do produto ou do polinômio expandido:
    #3x^2 - 7x + 4
    return 3*(x**2) - 7*x + 4

def phi22(x):
    #MPF para a raiz 1.5
    #isolando x da forma expandida x^3 - 3.5x^2 + 4x - 1.5 = 0
    #4x = -x^3 + 3.5x^2 + 1.5 => x = (3.5x^2 - x^3 + 1.5) / 4
    return (3.5*(x**2) - x**3 + 1.5) / 4

#paarametros: intervalo [1.1, 2.0] para focar na raiz 1.5 (onde há troca de sinal)
#se usar [0, 2], a raiz dupla em 1.0 atrapalha a bissecção.
executar_bonito("EXEMPLO 22", f22, df22, phi22, 1.1, 2.0, 1.6, 2.0, 1e-6)


                         EXEMPLO 22                         
Intervalo: [1.1, 2.0] | Precisão: 1e-06


,Método,Iterações (k),Raiz Encontrada,Erro Abs |f(x)|,Tempo (s)
0,Bissecção,16,1.49999847,0.00000038,0.00001060
1,Posição Falsa,55,1.49999679,0.00000080,0.00003230
2,MPF,100,1.50010848,0.00002713,0.00004080
3,Newton,4,1.50000000,0.00000000,0.00000450
4,Secante,6,1.50000042,0.00000010,0.00000320
